# 06 — Leakage-Safe Anomaly Detection & Disclosure Prediction

**Purpose:** Build the first reproducible modeling baseline from Notebook 05's **829,027 canonical Form 990 organization-year records**. The target is **self-reported significant diversion of assets**, **not verified fraud**. A negative label means no positive disclosure was extracted, **not** that the organization is confirmed clean.

**Outputs:** time/EIN-aware cohort diagnostics, leakage-reviewed feature manifest, unsupervised anomaly scores, supervised disclosure scores, precision/recall at investigation capacity, and saved artifacts. 

**Before running:** Review the detected label/year/EIN columns and the candidate feature list in Sections 2–4. Auto-detection is a convenience, **not a substitute for leakage review**. This notebook does not use supplemental 990-EZ/PF/T or cross-form link counts: publication/availability timing has not yet been validated.

In [ ]:
from pathlib import Path
import json
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    make_scorer,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import RandomOverSampler

In [ ]:
RANDOM_STATE = 42
MAX_FEATURES = 100
# Use None to run all rows; change to a positive integer for a fast dry run.
MAX_ROWS = None
# Candidate models are intentionally simple first baselines, not calibrated probabilities.
np.random.seed(RANDOM_STATE)
print('pandas', pd.__version__)

## 1. Load the Notebook 05 modeling dataset

Run this notebook from the repository root **or** from `notebooks/`. Paths are resolved relative to both locations. The original Parquet file is never modified.

In [ ]:
FILE = '05_form990_modeling_organization_year_v1.parquet'
candidates = [Path.cwd() / 'data/analytics/modeling' / FILE,
              Path.cwd().parent / 'data/analytics/modeling' / FILE]
source = next((p for p in candidates if p.exists()), None)
if source is None:
    raise FileNotFoundError(f'Could not locate {FILE}. Checked: {[str(p) for p in candidates]}')
OUT = source.parent / 'notebook_06_baselines'
raw = pd.read_parquet(source)
print('Source:', source)
print('Shape:', raw.shape)
print('First 60 columns:', raw.columns.tolist()[:60])
if MAX_ROWS is not None:
    raw = raw.sample(n=min(MAX_ROWS, len(raw)), random_state=RANDOM_STATE).copy()
    print('DRY RUN sample:', len(raw))

## 2. Confirm identifiers and the disclosure label

The default expected label is `fraud_label` from Notebook 05. **It denotes a self-reported significant diversion disclosure only.** Stop and inspect if any identifier is ambiguous.

In [ ]:
def choose_col(frame, preferred, alternatives=()):
    for name in (preferred, *alternatives):
        if name in frame.columns: return name
    raise KeyError(f'Missing {preferred}. Available columns: {frame.columns.tolist()}')

LABEL = choose_col(raw, 'fraud_label')
EIN = choose_col(raw, 'ein', ('EIN', 'organization_ein', 'org_ein'))
YEAR = choose_col(raw, 'tax_year', ('tax_yr', 'fiscal_year', 'return_year', 'year'))
print('Label:', LABEL, '| EIN:', EIN, '| Year:', YEAR)
print('Raw label counts:'); display(raw[LABEL].value_counts(dropna=False).to_frame('records'))
print('Raw year counts:'); display(raw[YEAR].value_counts(dropna=False).sort_index().to_frame('records'))
assert raw[LABEL].notna().all(), 'Missing labels: stop and investigate'
assert set(pd.to_numeric(raw[LABEL], errors='coerce').dropna().unique()).issubset({0,1}), 'Unexpected labels'
assert not raw.duplicated([EIN, YEAR]).any(), 'Multiple rows per EIN/tax year: stop'
assert raw[EIN].notna().all() and raw[YEAR].notna().all(), 'Missing EIN or tax year'
frame = raw.copy()
frame[LABEL] = pd.to_numeric(frame[LABEL]).astype('int8')
frame[YEAR] = pd.to_numeric(frame[YEAR], errors='raise').astype('int16')
frame[EIN] = frame[EIN].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(9)
assert not frame.duplicated([EIN, YEAR]).any(), 'Normalized EIN duplicates detected'
print('Validated', len(frame), 'organization-year records, positives:', frame[LABEL].sum())

## 3. Strict time-ordered and EIN-disjoint splits

**Training:** tax year 2023. **Validation:** tax year 2024, excluding any EIN in training. **Test:** tax year 2025, excluding any EIN in training or validation. This conservative design prevents an organization's history from appearing in multiple sets and uses earlier tax years to predict later ones.

**Trade-off:** It evaluates **previously unseen organizations** only, not all returning organizations. 2025 filings are incomplete, and 2025 is a potentially biased holdout. We report exclusions and positive counts explicitly. If the resulting splits have too few positives, **do not treat model comparisons as reliable**.

In [ ]:
expected_years = {2023, 2024, 2025}
print('All tax years:', sorted(frame[YEAR].unique().tolist()))
assert expected_years.issubset(set(frame[YEAR])), 'Expected 2023, 2024, and 2025 tax years'
train = frame.loc[frame[YEAR].eq(2023)].copy()
seen_train = set(train[EIN])
val_all = frame.loc[frame[YEAR].eq(2024)].copy()
val = val_all.loc[~val_all[EIN].isin(seen_train)].copy()
seen_val = set(val[EIN])
test_all = frame.loc[frame[YEAR].eq(2025)].copy()
test = test_all.loc[~test_all[EIN].isin(seen_train | seen_val)].copy()
assert not (set(train[EIN]) & set(val[EIN]))
assert not (set(train[EIN]) & set(test[EIN]))
assert not (set(val[EIN]) & set(test[EIN]))
summary = pd.DataFrame([
    {'split': name, 'rows': len(df), 'positive': int(df[LABEL].sum()), 'positive_rate': df[LABEL].mean(), 'unique_ein': df[EIN].nunique()}
    for name, df in [('train_2023',train),('validation_2024_unseen',val),('test_2025_unseen',test)]
])
display(summary)
print('2024 rows excluded due to 2023 EIN overlap:', len(val_all)-len(val))
print('2025 rows excluded due to earlier split EIN overlap:', len(test_all)-len(test))
assert min(len(train),len(val),len(test)) > 0, 'One split is empty'
assert all(df[LABEL].sum() > 0 for df in [train,val,test]), 'At least one split has no positive labels: evaluation is not possible'
if min(df[LABEL].sum() for df in [val,test]) < 20:
    warnings.warn('Validation/test have fewer than 20 positives; rankings and PR-AUC will be unstable.')

## 4. Feature screening and explicit leakage review

Only numeric columns are eligible. Identifier fields, target fields, diversion-related fields, audit/investigation outcomes, publication/filing metadata, and suspiciously named post-outcome features are excluded. **This is deliberately conservative.** Inspect the full manifest below before trusting any scores. Year-over-year changes can leak information if calculated using future tax years; we **exclude all change/trend/lag features for this first baseline** until their construction is verified to be strictly backward-looking.

Some numeric financial values may be known only when a return is published. This baseline is a **filing-time** risk model, not a pre-filing forecast.

In [ ]:
EXACT_BLOCK = {LABEL, EIN, YEAR, 'filing_id', 'publication_year', 'tax_period_end',
               'tax_period_begin', 'source_file', 'file_path', 'record_id', 'index'}
BLOCK_PATTERNS = [
    r'fraud', r'diversion', r'misappropriat', r'embezzl', r'loss_event', r'case_outcome',
    r'investigat', r'enforcement', r'whistleblow', r'restatement', r'ever_positive',
    r'positive_filing', r'label', r'target', r'outcome', r'future', r'next_year',
    r'publication', r'filing_id', r'object_id', r'row_id', r'(^|_)id$',
    r'change', r'growth', r'lag', r'lead', r'trend', r'rolling', r'prior_year',
    r'amend', r'return_type', r'tax_period', r'filing_date', r'filing_year',
]
def blocked(name):
    low = name.lower()
    return name in EXACT_BLOCK or any(re.search(p, low) for p in BLOCK_PATTERNS)

numeric = frame.select_dtypes(include='number').columns.tolist()
allowed = [c for c in numeric if not blocked(c)]
blocked_numeric = [c for c in numeric if blocked(c)]
# Remove columns entirely missing or constant in training only.
usable = [c for c in allowed if train[c].notna().any() and train[c].nunique(dropna=True)>1]
# Avoid an arbitrary subset: fail loudly if unexpectedly too many candidate features.
if len(usable) > MAX_FEATURES:
    raise ValueError(f'{len(usable)} features passed. Review and increase MAX_FEATURES intentionally; do not silently truncate.')
FEATURES = sorted(usable)
print('Allowed candidate features:', len(FEATURES))
print('Blocked numeric fields:', len(blocked_numeric))
print('SELECTED FEATURES — REVIEW EACH FOR LEAKAGE:')
for c in FEATURES: print('  ', c)
print('BLOCKED NUMERIC FIELDS:')
for c in blocked_numeric: print('  ', c)
# Exclude potentially leaky or redundant features
ADDITIONAL_EXCLUSIONS = {
    "filing_count",
    "financial_filing_count",
    "balance_sheet_residual",
    "full990_revenue_minus_expenses",
}

FEATURES = [f for f in FEATURES if f not in ADDITIONAL_EXCLUSIONS]

print(f"Final selected features: {len(FEATURES)}")
print("\n".join(FEATURES))

assert FEATURES, "No usable numeric features."

FEATURES_REVIEWED = True

### Step 4 Results: Feature Selection and Leakage Prevention

The initial screening identified **47 candidate features** and blocked **19 numeric fields**. After reviewing the candidates, four additional features were excluded due to potential data leakage or redundancy, leaving **43 features for modeling**.

#### Selected Feature Categories
- **Financial performance:** Revenue, expenses, assets, liabilities, operating margins, and financial ratios.
- **Governance:** Board membership, independence, employees, and volunteers.
- **Compensation:** Compensation totals and reporting completeness.
- **Financial reporting quality:** Balance sheet discrepancies and functional expense reconciliation differences.

#### Leakage Prevention
- Excluded fields directly related to the self-reported diversion disclosure label.
- Removed four additional potentially leaky or redundant features.
- The 19 initially blocked numeric fields include both potential leakage risks and fields excluded for other reasons.


#### Why 19 Numeric Fields Were Blocked

The initial feature screening excluded 19 numeric fields:
- **6 label-related fields:** Excluded to prevent direct or indirect disclosure of the prediction target.
- **10 year-over-year financial change fields:** Temporarily excluded until their calculations and timing can be validated.
- **3 date/publication fields:** Excluded as model inputs to avoid learning reporting-period patterns; retained where needed for dataset splitting.

These exclusions are intentionally conservative. Some fields, particularly year-over-year financial changes, may be reintroduced after additional leakage checks.


#### Feature Review Status and Remaining Limitations

The initial feature screening and manual review identified 43 features suitable for experimental modeling. Direct target-related fields, potentially problematic filing metadata, and selected redundant or potentially leaky variables were excluded.

The manual feature-review gate (`FEATURES_REVIEWED = True`) was acknowledged to allow the modeling experiments to proceed. However, this does not establish that every selected feature is available at the intended prediction time.

A complete point-in-time feature audit remains necessary before prospective deployment. In particular, financial and organizational information reported on Form 990 must not be treated as available before the relevant filing becomes accessible.

The current feature set is appropriate for exploratory modeling but should not be considered fully validated for prospective fraud detection or investigative prioritization.

**Next step:** Create organization-disjoint training, validation, and test splits to evaluate model performance on unseen organizations.


## 5. Train-only preprocessing

Median imputation and robust scaling are fit **only on 2023 training rows**. Validation and test data use those fitted transformations. We add missingness indicators to retain reporting-completeness signals; assess whether they reflect IRS publication artifacts before deployment.

In [ ]:
X_train, X_val, X_test = (df[FEATURES].replace([np.inf,-np.inf],np.nan) for df in [train,val,test])
y_train, y_val, y_test = (df[LABEL].to_numpy() for df in [train,val,test])
preprocessor = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
    ('scale', RobustScaler()),
])
Xt = preprocessor.fit_transform(X_train)
Xv = preprocessor.transform(X_val)
Xh = preprocessor.transform(X_test)
print('Transformed shape:', Xt.shape, Xv.shape, Xh.shape)
assert np.isfinite(Xt).all() and np.isfinite(Xv).all() and np.isfinite(Xh).all()

In [ ]:
# Positive disclosure counts
for name, data in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)
]:
    positives = int(data["fraud_label"].sum())
    print(f"{name}: {len(data):,} rows, {positives} positives")

# Check EIN overlap between splits
train_eins = set(train["ein"])
val_eins = set(val["ein"])
test_eins = set(test["ein"])

print("\nOrganization overlap:")
print("Train / Validation:", len(train_eins & val_eins))
print("Train / Test:", len(train_eins & test_eins))
print("Validation / Test:", len(val_eins & test_eins))

### Step 5 Results: Data Splitting and Leakage Prevention

The dataset was split chronologically into training (2023), validation (2024), and test (2025) groups, with organizations appearing in only one split.

| Dataset | Records | Positive Disclosures |
|---|---:|---:|
| Training | 352,798 | 316 |
| Validation | 26,114 | 35 |
| Test | 10,049 | 14 |

**Validation:** All organization overlap checks returned zero, confirming that no EIN appears in multiple splits.

**Limitation:** Because positive disclosures are extremely rare, particularly in the test set (14 positives), evaluation metrics may be unstable. Results should be treated as preliminary and interpreted alongside precision, recall, and investigative-capacity metrics.

**Next step:** Train the unsupervised anomaly detection and supervised classification baselines, after confirming preprocessing was fitted only on training data.


In [ ]:
# Oversample training data only
# 0.05 means positives will equal 5% of negatives
sampler = RandomOverSampler(
    sampling_strategy=0.05,
    random_state=42
)

Xt_oversampled, y_train_oversampled = sampler.fit_resample(
    Xt, y_train
)

print("Original training data:")
print("Total:", len(y_train))
print("Positive:", int(np.sum(y_train)))

print("\nOversampled training data:")
print("Total:", len(y_train_oversampled))
print("Positive:", int(np.sum(y_train_oversampled)))

print("\nValidation and test sets remain unchanged.")

## 6. Unsupervised anomaly detection

Isolation Forest learns unusual financial/reporting patterns **without using disclosure labels**. Its score is a ranking, not a fraud probability. We fit on a capped training sample for runtime and evaluate on untouched validation/test splits.

In [ ]:
ISO_TRAIN_CAP = 100_000
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(Xt), size=min(ISO_TRAIN_CAP,len(Xt)), replace=False)
iso = IsolationForest(n_estimators=200, max_samples=2048, contamination='auto',
                      n_jobs=-1, random_state=RANDOM_STATE)
iso.fit(Xt[idx])
# Higher means more anomalous.
iso_val = -iso.decision_function(Xv)
iso_test = -iso.decision_function(Xh)
print('Isolation Forest fitted on', len(idx), 'training rows')

## 7. Supervised disclosure prediction

A histogram gradient boosting model predicts the **disclosure label**, not proven fraud. Positive cases are very rare; we use train-only positive weighting and evaluate **precision-recall metrics**, not accuracy. Scores are uncalibrated ranking outputs.

In [ ]:
n_pos = int(y_train.sum()); n_neg = len(y_train)-n_pos
pos_weight = min(n_neg / max(n_pos,1), 500.0)
weights = np.where(y_train==1, pos_weight, 1.0)
clf = HistGradientBoostingClassifier(max_iter=120, learning_rate=0.05,
    max_leaf_nodes=15, l2_regularization=5.0, min_samples_leaf=50,
    random_state=RANDOM_STATE)
clf.fit(Xt, y_train, sample_weight=weights)
sup_val = clf.predict_proba(Xv)[:,1]
sup_test = clf.predict_proba(Xh)[:,1]
print('Training positives:',n_pos,'| positive sample weight:',round(pos_weight,2))

## 7A: Train supervised model using oversampled data

In [ ]:
# Same hyperparameters as Step 7 for a fair comparison

oversampled_model = HistGradientBoostingClassifier(
    max_iter=120,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=5.0,
    min_samples_leaf=50,
    random_state=RANDOM_STATE
)

# Use oversampled data WITHOUT additional class weights
oversampled_model.fit(
    Xt_oversampled,
    y_train_oversampled
)

# Evaluate on unchanged validation set
oversampled_val_scores = oversampled_model.predict_proba(Xv)[:, 1]

print(
    "Oversampled validation PR-AUC:",
    round(average_precision_score(y_val, oversampled_val_scores), 4)
)

for k in [25, 50, 100, 250, 500]:
    top_indices = np.argsort(oversampled_val_scores)[::-1][:k]
    found = int(np.asarray(y_val)[top_indices].sum())

    print(
        f"Top {k}: {found} positives | "
        f"Precision: {found/k:.2%} | "
        f"Recall: {found/int(np.sum(y_val)):.2%}"
    )

In [ ]:
def compare_model(name, scores):
    y = np.asarray(y_val)
    scores = np.asarray(scores)

    result = {
        "Model": name,
        "PR-AUC": average_precision_score(y, scores),
        "ROC-AUC": roc_auc_score(y, scores)
    }

    for k in [25, 50, 100, 250, 500]:
        top_idx = np.argsort(scores)[::-1][:k]
        found = int(y[top_idx].sum())

        result[f"Found@{k}"] = found
        result[f"Precision@{k}"] = found / k

    return result

results = pd.DataFrame([
    compare_model("Original weighted model", sup_val),
    compare_model("Oversampled model", oversampled_val_scores),
    compare_model("Isolation Forest", iso_val)
])

display(results.round(4))


## Step 7B: Compare class weights on the unchanged validation set

In [ ]:
weight_results = []
weight_models = {}

for positive_weight in [1, 10, 50, 100, 250, 500]:

    sample_weights = np.where(
        np.asarray(y_train) == 1,
        positive_weight,
        1.0
    )

    model = HistGradientBoostingClassifier(
        max_iter=120,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=5.0,
        min_samples_leaf=50,
        random_state=RANDOM_STATE
    )

    model.fit(Xt, y_train, sample_weight=sample_weights)

    scores = model.predict_proba(Xv)[:, 1]
    top100 = np.argsort(scores)[::-1][:100]
    top500 = np.argsort(scores)[::-1][:500]

    weight_results.append({
        "Positive weight": positive_weight,
        "PR-AUC": average_precision_score(y_val, scores),
        "Found@100": int(np.asarray(y_val)[top100].sum()),
        "Found@500": int(np.asarray(y_val)[top500].sum())
    })

    weight_models[positive_weight] = model

weight_comparison = pd.DataFrame(weight_results)
display(weight_comparison.sort_values("PR-AUC", ascending=False))

## Step 7C: Detailed validation ranking comparison

In [ ]:
rows = []
y = np.asarray(y_val)

for weight in [10, 50, 100, 250, 500]:
    scores = weight_models[weight].predict_proba(Xv)[:, 1]
    order = np.argsort(-scores, kind="stable")

    result = {
        "Weight": weight,
        "PR-AUC": average_precision_score(y, scores)
    }

    for k in [25, 50, 100, 250, 500, 1000]:
        found = int(y[order[:k]].sum())
        result[f"Found@{k}"] = found
        result[f"Recall@{k}"] = found / int(y.sum())

    rows.append(result)

detailed_comparison = pd.DataFrame(rows)
display(detailed_comparison.round(4))


## Step 7D: Examine where each model ranks positive disclosures

In [ ]:
y = np.asarray(y_val)
positive_indices = np.where(y == 1)[0]

ranking_results = []

for weight in [10, 50, 100, 250, 500]:

    scores = weight_models[weight].predict_proba(Xv)[:, 1]

    # Rank all filings from highest to lowest predicted risk
    order = np.argsort(-scores, kind="stable")

    # Assign each filing its rank
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)

    positive_ranks = np.sort(ranks[positive_indices])

    ranking_results.append({
        "Weight": weight,
        "Best positive rank": positive_ranks[0],
        "2nd positive rank": positive_ranks[1],
        "3rd positive rank": positive_ranks[2],
        "5th positive rank": positive_ranks[4],
        "Median positive rank": np.median(positive_ranks),
        "Worst positive rank": positive_ranks[-1]
    })

ranking_df = pd.DataFrame(ranking_results)

display(ranking_df)


## Step 7E: Feature importance using validation PR-AUC

In [ ]:
# Use weight-500 model as the original baseline
baseline_model = weight_models[500]

importance = permutation_importance(
    baseline_model,
    Xv,
    y_val,
    scoring="average_precision",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance.importances_mean,
    "Std": importance.importances_std
})

display(
    importance_df
    .sort_values("Importance", ascending=False)
    .head(20)
)



### Step 7E — Permutation Feature Importance

**Objective:** Identify which features contribute most to the supervised model's ability to distinguish Form 990 filings with self-reported significant diversions of assets from filings without a positive disclosure.

#### Methodology

Permutation importance was calculated using the original supervised HistGradientBoostingClassifier with a positive class weight of 500.

- **Dataset:** 2024 validation set (26,114 filings, 35 positive disclosures).
- **Features:** 67 preprocessed columns, including missing-value indicators.
- **Evaluation metric:** Average precision (PR-AUC).
- **Method:** Each feature was randomly shuffled three times while all other features remained unchanged.
- **Interpretation:** Importance measures the average decrease in validation PR-AUC after shuffling a feature. Larger positive values indicate greater contribution to model performance.

#### Key Findings

The highest-ranked features were:

| Feature | Importance | Interpretation |
|---|---:|---|
| `independent_board_members` | 0.003167 | Number of independent voting board members |
| `voting_board_members` | 0.003034 | Total voting board members |
| `part_vii_compensation_rows` | 0.002882 | Number of compensation records reported in Part VII |
| `assets_eoy` | 0.002566 | Total assets at end of year |
| `net_assets_boy` | 0.002476 | Net assets at beginning of year |
| `program_expense_share` | 0.002316 | Proportion of functional expenses allocated to programs |

**Governance and compensation features were among the strongest contributors.** Board composition and Part VII compensation reporting ranked above most individual financial metrics.

**Organization size may influence predictions.** Assets, net assets, revenue, and employee counts also contributed to model performance. These features may capture differences in organizational scale, reporting practices, or disclosure likelihood.

**Spending patterns provided additional signal.** Program expense share, compensation relative to expenses, and operating margin appeared among the top-ranked features, suggesting that resource allocation may help distinguish positive disclosures.

#### Limitations

1. **Small positive sample:** The validation set contains only 35 positive disclosures. Feature importance estimates may be unstable and disproportionately influenced by a few observations.
2. **Correlated features:** Related variables, such as independent board members and total voting board members, may share predictive information. Permutation importance can understate or distribute their individual contributions.
3. **No directionality:** Importance indicates whether a feature contributes to predictive performance, not whether higher or lower values are associated with greater disclosure risk.
4. **Disclosure versus fraud:** The target identifies self-reported significant diversions of assets, not independently verified fraud. The model may partially learn which organizations are more likely to detect and disclose incidents.
5. **Model-specific findings:** These results describe the weight-500 supervised model and should not automatically be generalized to other models or class weights.
6. **Validation reuse:** Permutation importance was calculated using the same 2024 validation dataset that informed model comparison and selection. These results are exploratory and may reflect validation-specific patterns. They should not be interpreted as independently validated explanations of model performance or causal relationships.


#### Conclusions and Next Steps

The current model appears to use a combination of governance structure, compensation reporting, organizational size, and financial allocation patterns.

Before introducing additional features, the next step is to compare the distributions of the highest-ranked variables between positive and negative training filings.

This analysis will help determine whether to prioritize:

- Board independence and governance ratios.
- Organization-size-adjusted financial metrics.
- Compensation and expense allocation features.
- Validated year-over-year financial changes.

**Overall conclusion:** Governance and organizational characteristics appear to contain useful predictive information, but the limited number of positive disclosures and weak overall model performance mean these findings should be treated as exploratory rather than definitive.


## Step 7F: Compare feature distributions by disclosure label

In [ ]:
# Training data only — do not use validation or test data

features_to_examine = [
    "independent_board_members",
    "voting_board_members",
    "part_vii_compensation_rows",
    "assets_eoy",
    "net_assets_boy",
    "program_expense_share",
    "revenue_py",
    "compensation_to_expenses",
    "employees_count",
    "operating_margin"
]

distribution_rows = []

for feature in features_to_examine:
    negative = pd.to_numeric(
        train.loc[train["fraud_label"] == 0, feature],
        errors="coerce"
    ).dropna()

    positive = pd.to_numeric(
        train.loc[train["fraud_label"] == 1, feature],
        errors="coerce"
    ).dropna()

    distribution_rows.append({
        "Feature": feature,
        "Negative median": negative.median(),
        "Positive median": positive.median(),
        "Negative non-null": len(negative),
        "Positive non-null": len(positive)
    })

distribution_df = pd.DataFrame(distribution_rows)

display(distribution_df.round(3))


## Step 7G: Compare feature distributions beyond the median

In [ ]:
# Training data only

features_to_examine = [
    "independent_board_members",
    "voting_board_members",
    "part_vii_compensation_rows",
    "assets_eoy",
    "employees_count",
    "program_expense_share",
    "operating_margin"
]

percentile_rows = []

for feature in features_to_examine:
    for label, label_name in [(0, "Negative"), (1, "Positive")]:
        values = pd.to_numeric(
            train.loc[train["fraud_label"] == label, feature],
            errors="coerce"
        ).dropna()

        percentile_rows.append({
            "Feature": feature,
            "Group": label_name,
            "P10": values.quantile(0.10),
            "P25": values.quantile(0.25),
            "Median": values.median(),
            "P75": values.quantile(0.75),
            "P90": values.quantile(0.90)
        })

percentile_df = pd.DataFrame(percentile_rows)
display(percentile_df.round(3))

### Step 7G — Visualization 1: End-of-year assets

In [ ]:
assets_plot = (
    percentile_df[
        percentile_df["Feature"] == "assets_eoy"
    ]
    .set_index("Group")
)

percentiles = ["P25", "Median", "P75", "P90"]
x = np.arange(len(percentiles))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    x - width / 2,
    assets_plot.loc["Negative", percentiles].astype(float).values / 1e6,
    width,
    label="Negative filings",
    color="#6488B8"
)

ax.bar(
    x + width / 2,
    assets_plot.loc["Positive", percentiles].astype(float).values / 1e6,
    width,
    label="Positive disclosures",
    color="#E58B64"
)

ax.set_xticks(x)
ax.set_xticklabels(percentiles)
ax.set_ylabel("End-of-year assets ($ millions)")
ax.set_title("End-of-Year Assets by Disclosure Group")
ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### Step 7G — Visualization 2: Operating margins

In [ ]:
margin_plot = (
    percentile_df[
        percentile_df["Feature"] == "operating_margin"
    ]
    .set_index("Group")
)

percentiles = ["P10", "P25", "Median", "P75", "P90"]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    percentiles,
    margin_plot.loc["Negative", percentiles].astype(float).values * 100,
    marker="o",
    linewidth=2,
    label="Negative filings",
    color="#6488B8"
)

ax.plot(
    percentiles,
    margin_plot.loc["Positive", percentiles].astype(float).values * 100,
    marker="o",
    linewidth=2,
    label="Positive disclosures",
    color="#E58B64"
)

ax.axhline(0, color="gray", linestyle="--", alpha=0.6)

ax.set_ylabel("Operating margin (%)")
ax.set_title("Operating Margin by Disclosure Group")
ax.legend()
ax.grid(alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


### Step 7G — Feature Distribution Analysis: Percentiles and Visualizations

**Objective:** Examine differences in feature distributions between organizations with and without self-reported significant diversions of assets, beyond the median comparisons performed in Step 7F.

#### Methodology

Using the 2023 training dataset, the 10th, 25th, 50th (median), 75th, and 90th percentiles were calculated separately for positive and negative disclosures.

Seven features were examined:
- Independent board members
- Voting board members
- Part VII compensation rows
- End-of-year assets
- Employee count
- Program expense share
- Operating margin

The analysis uses training data only and does not modify the model or use the held-out test set.

#### Key Findings

**1. Governance and compensation distributions**

The median number of independent board members (7), voting board members (8), and Part VII compensation rows (9) was identical between positive and negative filings.

However, positive-disclosure organizations had fewer independent board members at the 25th percentile (2 versus 4).

This suggests that governance differences may exist within specific portions of the distribution even when overall medians are similar.

**2. Organizational assets**

Positive-disclosure organizations had lower end-of-year assets at the 25th, 50th, and 75th percentiles.

- Median assets: approximately $544,000 for positive disclosures versus $810,000 for negative filings.
- 90th percentile assets: approximately $28.2 million for positive disclosures versus $15.8 million for negative filings.

This suggests greater variation in organization size among positive-disclosure filings. Organizational scale may interact with governance, compensation, and financial characteristics.

**3. Employee counts**

Positive-disclosure organizations generally reported more employees.

- Median: 4 employees versus 2.
- 75th percentile: 21 versus 15.
- 90th percentile: 102 versus 68.

Employee count may capture aspects of operational complexity not fully reflected by financial assets.

**4. Program expense allocation**

Positive-disclosure organizations had lower program expense shares across most examined percentiles.

- Median: 83.9% versus 86.7%.
- 25th percentile: 71.9% versus 75.0%.
- 75th percentile: 92.5% versus 96.4%.

This indicates a modest difference in how reported functional expenses are allocated.

**5. Operating margins**

Positive-disclosure organizations had lower operating margins at every examined percentile.

- 10th percentile: -49.5% versus -32.0%.
- 25th percentile: -16.1% versus -7.7%.
- Median: 1.6% versus 3.7%.
- 90th percentile: 32.7% versus 44.8%.

This is one of the more consistent descriptive differences observed in the analysis and suggests that financial performance may warrant further investigation.

#### Interpretation and Limitations

These results describe associations, not causal relationships.

The positive training group contains only 316 filings, compared with 352,482 negative filings. Consequently, positive-group percentiles are less stable and may be influenced by extreme observations.

Additionally, the target represents self-reported significant diversions of assets, not independently verified fraud. Differences may reflect reporting practices, organizational characteristics, or the ability to detect and disclose incidents.

Percentiles were calculated from non-null values, so sample sizes differ across financial features.

#### Conclusions and Next Steps

The analysis identifies several promising areas for feature engineering:

1. **Governance ratios:** Board independence relative to total voting board members.
2. **Organization-size adjustments:** Financial amounts normalized by employee counts or other measures of scale.
3. **Financial performance:** Operating deficit indicators and related financial ratios.
4. **Financial transformations:** Log transformations to reduce skewness in financial amounts.

The next stage will introduce selected features in Step 8 and compare their predictive value against the original 43-feature supervised model.

**Overall conclusion:** Financial performance, organizational size, and certain governance characteristics show potentially informative differences between disclosure groups. Further modeling is necessary to determine whether these differences improve out-of-sample detection.


## Step 8 — Feature Engineering and Model Improvement

### Step 8A — Create Additional Predictive Features

**Objective:** Engineer interpretable features that capture nonprofit governance, financial stability, operating scale, and spending patterns.

The initial feature analysis identified differences in operating margins, program expense allocation, employee counts, and organizational assets between positive and negative disclosure groups.

This experiment introduces:
- Board independence ratio
- Assets and revenue per employee
- Operating deficit indicator
- Low program expense share indicator
- Signed logarithmic transformations of financial amounts

All new features are calculated independently within each filing. No future-year information or disclosure labels are used to construct predictors.

The original feature set and fitted models are preserved for comparison.


## Step 8A: Feature engineering

In [ ]:
# Preserve original datasets
train_fe = train.copy()
val_fe = val.copy()

def engineer_features(df):
    df = df.copy()

    # 1. Board independence ratio
    voting = pd.to_numeric(
        df["voting_board_members"], errors="coerce"
    )
    independent = pd.to_numeric(
        df["independent_board_members"], errors="coerce"
    )

    df["board_independence_ratio"] = (
        independent / voting.replace(0, np.nan)
    )

    # 2. Financial amounts per employee
    employees = pd.to_numeric(
        df["employees_count"], errors="coerce"
    ).replace(0, np.nan)

    assets = pd.to_numeric(
        df["assets_eoy"], errors="coerce"
    )

    revenue = pd.to_numeric(
        df["revenue_cy"], errors="coerce"
    )

    df["assets_per_employee"] = assets / employees
    df["revenue_per_employee"] = revenue / employees

    # 3. Operating deficit indicator
    margin = pd.to_numeric(
        df["operating_margin"], errors="coerce"
    )

    df["operating_deficit_flag"] = (
        margin < 0
    ).astype(float)

    df.loc[
        margin.isna(), "operating_deficit_flag"
    ] = np.nan

    # 4. Low program expense share indicator
    program_share = pd.to_numeric(
        df["program_expense_share"], errors="coerce"
    )

    # Exploratory threshold, not a fraud benchmark
    df["low_program_expense_flag"] = (
        program_share < 0.75
    ).astype(float)

    df.loc[
        program_share.isna(), "low_program_expense_flag"
    ] = np.nan

    # 5. Signed logarithmic financial transformations
    # Preserves the sign of negative values
    log_features = [
        "assets_eoy",
        "net_assets_boy",
        "revenue_cy",
        "expenses_cy"
    ]

    for col in log_features:
        values = pd.to_numeric(
            df[col], errors="coerce"
        )

        df[f"log_{col}"] = (
            np.sign(values) * np.log1p(np.abs(values))
        )

    return df


train_fe = engineer_features(train_fe)
val_fe = engineer_features(val_fe)

NEW_FEATURES = [
    "board_independence_ratio",
    "assets_per_employee",
    "revenue_per_employee",
    "operating_deficit_flag",
    "low_program_expense_flag",
    "log_assets_eoy",
    "log_net_assets_boy",
    "log_revenue_cy",
    "log_expenses_cy"
]

print("New features created:", len(NEW_FEATURES))
print("Training shape:", train_fe.shape)
print("Validation shape:", val_fe.shape)

display(train_fe[NEW_FEATURES].head())

## Step 8B: Feature quality checks

In [ ]:
quality_rows = []

for feature in NEW_FEATURES:
    values = pd.to_numeric(
        train_fe[feature], errors="coerce"
    )

    quality_rows.append({
        "Feature": feature,
        "Missing count": int(values.isna().sum()),
        "Missing %": round(values.isna().mean() * 100, 2),
        "Infinite count": int(np.isinf(values).sum()),
        "Minimum": values.min(),
        "Median": values.median(),
        "Maximum": values.max()
    })

feature_quality = pd.DataFrame(quality_rows)

display(feature_quality)

assert len(train_fe) == len(train)
assert len(val_fe) == len(val)

# Check for unexpected infinite values
for df in [train_fe, val_fe]:
    values = df[NEW_FEATURES].to_numpy(dtype=float)
    assert not np.isinf(values).any(), (
        "Infinite values found in engineered features"
    )

print("Feature quality checks passed.")

In [ ]:
# Refine engineered features

def refine_engineered_features(df):
    df = df.copy()

    # Flag inconsistent board counts
    ratio = df["board_independence_ratio"]

    df["invalid_board_ratio_flag"] = (
        (ratio < 0) | (ratio > 1)
    ).astype(int)

    # Invalid ratios are treated as missing
    df.loc[
        (ratio < 0) | (ratio > 1),
        "board_independence_ratio"
    ] = np.nan

    # Log-transform per-employee financial ratios
    for col in [
        "assets_per_employee",
        "revenue_per_employee"
    ]:
        values = df[col]

        df[f"log_{col}"] = (
            np.sign(values) * np.log1p(np.abs(values))
        )

    return df


train_fe = refine_engineered_features(train_fe)
val_fe = refine_engineered_features(val_fe)

# Replace raw per-employee ratios with transformed versions
NEW_FEATURES = [
    col for col in NEW_FEATURES
    if col not in [
        "assets_per_employee",
        "revenue_per_employee"
    ]
]

NEW_FEATURES += [
    "invalid_board_ratio_flag",
    "log_assets_per_employee",
    "log_revenue_per_employee"
]

print("Final engineered feature count:", len(NEW_FEATURES))

display(
    train_fe[NEW_FEATURES]
    .describe()
    .T[["count", "min", "50%", "max"]]
)


In [ ]:
# Validate refined features

print(
    "Training filings with invalid board ratios:",
    int(train_fe["invalid_board_ratio_flag"].sum())
)

print(
    "Validation filings with invalid board ratios:",
    int(val_fe["invalid_board_ratio_flag"].sum())
)

print(
    "Missing employee-adjusted financial ratios:",
    round(
        train_fe["log_assets_per_employee"].isna().mean() * 100,
        2
    ),
    "%"
)

valid_board_ratios = train_fe[
    "board_independence_ratio"
].dropna()

assert valid_board_ratios.between(0, 1).all()

assert not np.isinf(
    train_fe[NEW_FEATURES].to_numpy(dtype=float)
).any()

assert not np.isinf(
    val_fe[NEW_FEATURES].to_numpy(dtype=float)
).any()

print("Refined feature checks passed.")

## Step 8C: Build original and expanded feature matrices

In [ ]:
ORIGINAL_FEATURES = list(X_train.columns)

EXPANDED_FEATURES = ORIGINAL_FEATURES + [
    col for col in NEW_FEATURES
    if col not in ORIGINAL_FEATURES
]

print("Original features:", len(ORIGINAL_FEATURES))
print("Expanded features:", len(EXPANDED_FEATURES))

assert len(ORIGINAL_FEATURES) == 43
assert len(EXPANDED_FEATURES) == 53
assert len(set(EXPANDED_FEATURES)) == len(EXPANDED_FEATURES)

X_train_original_8 = train_fe[ORIGINAL_FEATURES].copy()
X_val_original_8 = val_fe[ORIGINAL_FEATURES].copy()

X_train_expanded = train_fe[EXPANDED_FEATURES].copy()
X_val_expanded = val_fe[EXPANDED_FEATURES].copy()

print("Original training:", X_train_original_8.shape)
print("Expanded training:", X_train_expanded.shape)
print("Expanded validation:", X_val_expanded.shape)

In [ ]:
# Preprocess original and expanded features
# Match the original preprocessing pipeline exactly
# Fit preprocessing on training data only

# Recreate the original pipeline with identical settings
preprocessor_original_8 = clone(preprocessor)
preprocessor_expanded = clone(preprocessor)

# Original 43 features
Xt_original_8 = preprocessor_original_8.fit_transform(
    X_train_original_8
)

Xv_original_8 = preprocessor_original_8.transform(
    X_val_original_8
)

# Expanded 53 features
Xt_expanded = preprocessor_expanded.fit_transform(
    X_train_expanded
)

Xv_expanded = preprocessor_expanded.transform(
    X_val_expanded
)

print("Original transformed training:", Xt_original_8.shape)
print("Original transformed validation:", Xv_original_8.shape)

print("Expanded transformed training:", Xt_expanded.shape)
print("Expanded transformed validation:", Xv_expanded.shape)


In [ ]:
# Verify preprocessing consistency

train_match = (
    Xt.shape == Xt_original_8.shape
    and np.allclose(
        Xt, Xt_original_8,
        equal_nan=True
    )
)

val_match = (
    Xv.shape == Xv_original_8.shape
    and np.allclose(
        Xv, Xv_original_8,
        equal_nan=True
    )
)

print("Original training matrix matches:", train_match)
print("Original validation matrix matches:", val_match)

print("\nOriginal training shape:", Xt.shape)
print("Rebuilt training shape:", Xt_original_8.shape)

print("\nExpanded training shape:", Xt_expanded.shape)
print("Expanded validation shape:", Xv_expanded.shape)

assert train_match, "Training preprocessing mismatch"
assert val_match, "Validation preprocessing mismatch"

print("\nStep 8C preprocessing verified successfully.")

## Step 8D: Controlled feature engineering experiment

In [ ]:
experiment_models = {}
experiment_scores = {}
experiment_results = []

labels = np.asarray(y_val)

for weight in [50, 500]:

    sample_weights = np.where(
        np.asarray(y_train) == 1,
        weight,
        1.0
    )

    for feature_set, Xtr, Xvl in [
        ("Original", Xt_original_8, Xv_original_8),
        ("Expanded", Xt_expanded, Xv_expanded)
    ]:

        print(f"Training {feature_set} features, weight={weight}")

        model = HistGradientBoostingClassifier(
            max_iter=120,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=5.0,
            min_samples_leaf=50,
            random_state=RANDOM_STATE
        )

        model.fit(
            Xtr,
            y_train,
            sample_weight=sample_weights
        )

        scores = model.predict_proba(Xvl)[:, 1]
        order = np.argsort(-scores, kind="stable")

        key = (feature_set, weight)

        experiment_models[key] = model
        experiment_scores[key] = scores

        result = {
            "Feature set": feature_set,
            "Weight": weight,
            "PR-AUC": average_precision_score(
                labels, scores
            ),
            "ROC-AUC": roc_auc_score(
                labels, scores
            )
        }

        for k in [25, 50, 100, 250, 500, 1000]:
            result[f"Found@{k}"] = int(
                labels[order[:k]].sum()
            )

        experiment_results.append(result)

experiment_df = pd.DataFrame(experiment_results)

display(
    experiment_df.sort_values(
        ["Weight", "Feature set"]
    ).round(5)
)

## Step 8E: Quantify changes from feature engineering

In [ ]:
comparison_rows = []

for weight in [50, 500]:

    original = experiment_df[
        (experiment_df["Feature set"] == "Original") &
        (experiment_df["Weight"] == weight)
    ].iloc[0]

    expanded = experiment_df[
        (experiment_df["Feature set"] == "Expanded") &
        (experiment_df["Weight"] == weight)
    ].iloc[0]

    comparison_rows.append({
        "Weight": weight,
        "Original PR-AUC": original["PR-AUC"],
        "Expanded PR-AUC": expanded["PR-AUC"],
        "PR-AUC Change %": (
            (expanded["PR-AUC"] / original["PR-AUC"]) - 1
        ) * 100,
        "ROC-AUC Change": (
            expanded["ROC-AUC"] - original["ROC-AUC"]
        ),
        "Found@100 Change": (
            expanded["Found@100"] - original["Found@100"]
        ),
        "Found@500 Change": (
            expanded["Found@500"] - original["Found@500"]
        ),
        "Found@1000 Change": (
            expanded["Found@1000"] - original["Found@1000"]
        )
    })

improvement_df = pd.DataFrame(comparison_rows)

display(improvement_df.round(4))

In [ ]:
# Visualize original vs expanded model performance

plot_df = experiment_df.pivot(
    index="Weight",
    columns="Feature set",
    values="PR-AUC"
)

ax = plot_df[["Original", "Expanded"]].plot(
    kind="bar",
    figsize=(9, 5),
    rot=0
)

ax.set_title("Validation PR-AUC: Original vs Engineered Features")
ax.set_xlabel("Positive Class Weight")
ax.set_ylabel("PR-AUC (Average Precision)")
ax.legend(title="Feature Set")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


### Feature Engineering Results and Conclusions

#### Objective

Evaluate whether adding 10 engineered financial and governance features improves the model's ability to rank nonprofit filings with self-reported significant diversions of assets.

#### Methodology

The original 43-feature dataset was compared against an expanded 53-feature dataset.

New features included board independence ratios, employee-adjusted financial metrics, financial performance indicators, logarithmic financial transformations, and a board reporting inconsistency flag.

Both feature sets used identical preprocessing:
- Median imputation with missingness indicators
- RobustScaler
- Training-only preprocessing fitting

HistGradientBoostingClassifier models were trained using identical hyperparameters and positive class weights of 50 and 500.

Performance was evaluated using the organization-disjoint 2024 validation dataset containing 26,114 filings and 35 positive disclosures.

#### Results

**Weight 50:**
- Original PR-AUC: 0.03036
- Expanded PR-AUC: 0.00533
- PR-AUC decreased approximately 82.4%.
- Both models identified one positive disclosure in the top 100 and two in the top 500.

**Weight 500:**
- Original PR-AUC: 0.00537
- Expanded PR-AUC: 0.00227
- PR-AUC decreased approximately 57.7%.
- The original model identified three positive disclosures in the top 500, compared with two for the expanded model.
- Both models identified three positive disclosures in the top 1,000.

#### Interpretation

The engineered features did not improve validation performance.

Although some engineered features captured descriptive differences between disclosure groups, those differences did not translate into improved predictive ranking.

Potential explanations include:
- Redundancy with existing financial and governance predictors.
- High missingness in employee-adjusted financial ratios.
- Noisy or inconsistent reporting in certain governance fields.
- Limited positive training examples.
- Model sensitivity to additional predictors.

These are hypotheses rather than established explanations.

#### Model Selection Decision

Retain the original 43-feature models as the leading candidates.

- **Weight 50:** Highest validation PR-AUC, but strongly influenced by one highly ranked positive disclosure.
- **Weight 500:** Better investigative yield at a capacity of 500 filings.

Do not promote the expanded models based on the current validation results.

#### Feature Engineering Limitations

The additional engineered features were evaluated using the existing training and 2024 validation datasets. None demonstrated sufficient improvement to justify replacing the original 43-feature model.

However, these experiments did not include a separate point-in-time audit of the engineered features. Before using any of these features in future models, their calculations and source data must be reviewed to ensure they would have been available at the intended prediction time.

The 2024 validation results support retaining the original feature set for this experiment, but they do not establish that the original or engineered features are fully validated for prospective use.

Future feature-engineering experiments should also use an appropriate validation strategy to reduce the risk of overfitting to the existing validation dataset.

#### Limitations

The validation set contains only 35 positive disclosures, making performance estimates unstable.

The target identifies self-reported significant diversion disclosures, not confirmed fraud. Models may learn disclosure patterns rather than underlying diversion risk.

Feature engineering decisions were informed by exploratory analyses, so independent holdout evaluation remains necessary.

#### Next Steps

Proceed to Step 9 to evaluate precision, recall, PR-AUC, and investigative capacity for the retained models.

Reserve the 2025 holdout dataset for final evaluation after selecting the model configuration.


## 9. Precision, recall, PR-AUC and investigative capacity

**Precision@K:** fraction of top K ranked filings with a positive disclosure. **Recall@K:** fraction of all positive disclosures captured in those K. **PR-AUC / average precision** should be compared with the prevalence baseline. Evaluate several realistic review capacities. All test metrics are descriptive; the 2025 cohort is incomplete and restricted to unseen EINs.

In [ ]:
def rank_metrics(y, scores, split, model, capacities=(25,50,100,250,500,1000)):
    y=np.asarray(y); scores=np.asarray(scores)
    if not np.isfinite(scores).all(): raise ValueError('Nonfinite model scores')
    rows=[]; order=np.argsort(-scores,kind='stable'); n_pos=int(y.sum())
    ap=average_precision_score(y,scores) if n_pos else np.nan
    auc=roc_auc_score(y,scores) if len(np.unique(y))==2 else np.nan
    for k in capacities:
        k=min(k,len(y)); found=int(y[order[:k]].sum())
        rows.append(dict(split=split,model=model,k=k,positives_found=found,
                         precision_at_k=found/k,recall_at_k=found/n_pos if n_pos else np.nan,
                         average_precision=ap,roc_auc=auc,base_rate=y.mean(),total_positives=n_pos))
    return rows

In [ ]:
# Evaluate models on 2024 validation data only

validation_models = {
    "isolation_forest": iso_val,
    "supervised_weight_50": experiment_scores[("Original", 50)],
    "supervised_weight_500": experiment_scores[("Original", 500)],
    "supervised_oversampled": oversampled_val_scores
}

rows = []

for model_name, scores in validation_models.items():
    rows += rank_metrics(
        y=y_val,
        scores=scores,
        split="validation_2024_unseen",
        model=model_name
    )

metrics = pd.DataFrame(rows)

display(
    metrics.sort_values(
        ["model", "k"]
    ).round(5)
)

In [ ]:
# Model comparison summary

summary = (
    metrics.pivot(
        index="model",
        columns="k",
        values="positives_found"
    )
    .rename(columns=lambda k: f"Found@{k}")
)

overall = (
    metrics.groupby("model")
    .first()[
        ["average_precision", "roc_auc", "base_rate", "total_positives"]
    ]
)

summary = overall.join(summary)

summary["AP lift"] = (
    summary["average_precision"] / summary["base_rate"]
)

display(
    summary.sort_values(
        "average_precision",
        ascending=False
    ).round(5)
)


In [ ]:
# Investigative capacity comparison

fig, ax = plt.subplots(figsize=(10, 6))

for model_name, group in metrics.groupby("model"):
    group = group.sort_values("k")

    ax.plot(
        group["k"],
        group["positives_found"],
        marker="o",
        linewidth=2,
        label=model_name
    )

ax.set_title(
    "Positive Disclosures Identified by Investigative Capacity"
)
ax.set_xlabel("Number of Filings Investigated (Top K)")
ax.set_ylabel("Positive Disclosures Found")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


### Model Evaluation and Investigative Capacity: Findings

#### Objective

Evaluate how effectively the anomaly detection and supervised learning models prioritize nonprofit Form 990 filings with self-reported significant diversion disclosures.

Rather than relying solely on traditional classification metrics, this analysis evaluates the number of positive disclosures identified within fixed investigative capacities.

#### Evaluation Dataset

- Validation year: 2024
- Organization-disjoint validation filings: 26,114
- Positive disclosures: 35
- Disclosure prevalence: approximately 0.134%
- Negative filings: 26,079

The 2025 holdout test set was not used during this evaluation.

#### Models Evaluated

1. Isolation Forest — unsupervised anomaly detection.
2. HistGradientBoostingClassifier — positive class weight 50.
3. HistGradientBoostingClassifier — positive class weight 500.
4. HistGradientBoostingClassifier — training-only random oversampling.

All supervised models used the original 43-feature dataset.

#### Results

| Model | PR-AUC | ROC-AUC | Found@100 | Found@500 | Found@1000 |
|---|---:|---:|---:|---:|---:|
| Weight 50 | 0.03036 | 0.55433 | 1 | 2 | 2 |
| Weight 500 | 0.00537 | 0.60681 | 1 | 3 | 3 |
| Oversampled | 0.00293 | 0.61805 | 0 | 3 | 5 |
| Isolation Forest | 0.00184 | 0.52650 | 0 | 2 | 3 |

#### Key Findings

**1. Investigative capacity affects model selection.**

At 25–100 investigations, the weighted supervised models identified one positive disclosure each.

At 250–500 investigations, the weight-500 model performed best or tied for best.

At 1,000 investigations, the oversampled model identified five positive disclosures, outperforming the other models.

**2. High PR-AUC does not necessarily indicate strong investigative performance.**

The weight-50 model achieved the highest PR-AUC (0.03036), but this result was strongly influenced by a single positive disclosure ranked first.

The oversampled model had a much lower PR-AUC but identified more positive disclosures within the top 1,000 filings.

This demonstrates the importance of evaluating ranking models at realistic investigative capacities.

**3. Performance exceeds random-selection prevalence but remains limited.**

The oversampled model achieved precision@1000 of 0.50%, compared with the overall validation disclosure prevalence of approximately 0.134%.

This represents approximately 3.73 times the prevalence expected under random selection.

However, the model identified only five of the 35 positive disclosures, corresponding to recall@1000 of 14.29%.

**4. Unsupervised anomaly detection was less effective than the strongest supervised alternatives.**

Isolation Forest achieved a PR-AUC of 0.00184 and identified three positive disclosures in the top 1,000 filings.

Although this exceeds the validation base rate in average precision, it did not outperform the strongest supervised models at the tested capacities.

#### Model Selection

The following models are retained for final holdout evaluation:

- **Weight-500 supervised model:** Preferred candidate for investigative capacities of approximately 250–500 filings.
- **Oversampled supervised model:** Preferred candidate for a capacity of approximately 1,000 filings.

The weight-50 model will be retained as a PR-AUC benchmark, and Isolation Forest as an unsupervised benchmark.

These selections are based exclusively on validation performance and will not be revised using the 2025 holdout results.

#### Limitations

The validation dataset contains only 35 positive disclosures, creating substantial uncertainty around performance estimates.

The target represents self-reported significant diversions of assets, not independently verified fraud.

Negative filings are not necessarily confirmed clean organizations.

The models may identify patterns associated with reporting and disclosure rather than underlying fraudulent activity.

The analysis does not establish whether reviewing a model-selected filing would uncover previously unknown misconduct.

#### Conclusion

The models demonstrate limited but measurable ability to concentrate positive disclosures among higher-ranked filings.

No single model dominates across all investigative capacities.

For this project, precision@K, recall@K, and investigative yield are more directly actionable than PR-AUC alone.

The next step is to evaluate the frozen model configurations against the untouched 2025 holdout dataset.

## Step 10: Prepare frozen models for holdout evaluation

In [ ]:
# No fitting, tuning, or model selection on the test set

# Verify the test matrix matches the original model's feature space
assert Xh.shape[1] == Xt.shape[1]

# Score using models already fitted on training data
test_model_scores = {
    "isolation_forest": iso_test,
    "supervised_weight_50": weight_models[50].predict_proba(Xh)[:, 1],
    "supervised_weight_500": weight_models[500].predict_proba(Xh)[:, 1],
    "supervised_oversampled": oversampled_model.predict_proba(Xh)[:, 1]
}

for model_name, scores in test_model_scores.items():
    assert len(scores) == len(y_test)
    assert np.isfinite(scores).all()

print("All four frozen models scored successfully.")
print("Test filings:", len(y_test))
print("Positive disclosures:", int(np.asarray(y_test).sum()))

In [ ]:
# Final holdout evaluation

test_rows = []

for model_name, scores in test_model_scores.items():
    test_rows += rank_metrics(
        y=y_test,
        scores=scores,
        split="test_2025_unseen",
        model=model_name
    )

test_metrics = pd.DataFrame(test_rows)

test_summary = (
    test_metrics.pivot(
        index="model",
        columns="k",
        values="positives_found"
    )
    .rename(columns=lambda k: f"Found@{k}")
)

test_overall = (
    test_metrics.groupby("model")
    .first()[
        ["average_precision", "roc_auc", "base_rate", "total_positives"]
    ]
)

test_summary = test_overall.join(test_summary)

test_summary["AP lift"] = (
    test_summary["average_precision"] /
    test_summary["base_rate"]
)

display(
    test_summary.sort_values(
        "average_precision",
        ascending=False
    ).round(5)
)


In [ ]:
# Compare validation and holdout performance

validation_overall = (
    metrics.groupby("model")
    .first()[
        ["average_precision", "roc_auc"]
    ]
    .rename(columns={
        "average_precision": "Validation PR-AUC",
        "roc_auc": "Validation ROC-AUC"
    })
)

holdout_overall = (
    test_metrics.groupby("model")
    .first()[
        ["average_precision", "roc_auc"]
    ]
    .rename(columns={
        "average_precision": "Test PR-AUC",
        "roc_auc": "Test ROC-AUC"
    })
)

generalization = validation_overall.join(holdout_overall)

generalization["PR-AUC Change"] = (
    generalization["Test PR-AUC"] -
    generalization["Validation PR-AUC"]
)

display(generalization.round(5))



### Step 10D — Final Holdout Evaluation: Findings and Conclusions

#### Objective

Evaluate the previously trained and selected models on the untouched 2025 holdout dataset to assess their ability to generalize beyond the 2023 training and 2024 validation populations.

No models were retrained or tuned using 2025 outcomes.

#### Holdout Dataset

- Test year: 2025
- Organization-disjoint filings: 10,049
- Positive disclosures: 14
- Negative filings: 10,035
- Positive disclosure prevalence: approximately 0.139%

The 2025 dataset represents an incomplete filing year.

#### Results

| Model | PR-AUC | ROC-AUC | Found@100 | Found@500 | Found@1000 |
|---|---:|---:|---:|---:|---:|
| Isolation Forest | 0.00386 | 0.69904 | 0 | 4 | 5 |
| Oversampled supervised | 0.00351 | 0.74586 | 0 | 2 | 3 |
| Supervised, weight 500 | 0.00322 | 0.70137 | 0 | 1 | 2 |
| Supervised, weight 50 | 0.00304 | 0.70416 | 0 | 1 | 3 |

#### Key Findings

**1. Isolation Forest achieved the strongest investigative yield on the holdout dataset.**

Isolation Forest identified four positive disclosures within the top 500 filings and five within the top 1,000.

At a capacity of 500 investigations:
- Precision@500: 0.80%
- Recall@500: 28.57%
- Enrichment over holdout prevalence: approximately 5.74x

However, these estimates are based on only 14 positive disclosures and should be interpreted cautiously.

**2. Validation model rankings did not consistently generalize.**

The weight-500 supervised model identified three positives within the top 500 validation filings but only one within the top 500 holdout filings.

The oversampled model identified three within the top 500 validation filings and two within the top 500 holdout filings.

Isolation Forest showed the opposite pattern, identifying two within the top 500 validation filings and four within the top 500 holdout filings.

Differences in sample size, filing composition, and the small number of positive disclosures limit direct comparisons.

**3. The weight-50 model's validation PR-AUC was unstable.**

Its PR-AUC declined from 0.03036 on validation to 0.00304 on holdout.

The unusually high validation PR-AUC was influenced by a single positive disclosure ranked first, rather than consistently strong ranking performance.

**4. ROC-AUC and investigative yield capture different aspects of performance.**

The oversampled model achieved the highest holdout ROC-AUC (0.74586), indicating the strongest overall discrimination among the evaluated models.

However, Isolation Forest identified more positive disclosures within the top 500 and top 1,000 filings.

This reinforces the importance of reporting investigative capacity metrics alongside conventional model evaluation metrics.

#### Model Selection and Generalization

The original validation-based selections remain unchanged:

- Weight-500 supervised model for approximately 250–500 investigations.
- Oversampled supervised model for approximately 1,000 investigations.

Isolation Forest's stronger holdout results are reported as an independent evaluation finding, not used to retrospectively select or tune the model.

#### Limitations

The holdout dataset contains only 14 positive disclosures, making performance estimates highly uncertain.

The 2025 filing year is incomplete, and the test population may not represent the full-year filing population.

The target measures self-reported significant diversions of assets rather than confirmed fraud. A negative label does not establish the absence of misconduct.

The models may capture disclosure behavior, reporting characteristics, or organizational differences rather than underlying fraudulent activity.

The results do not establish operational effectiveness in identifying previously undisclosed diversion incidents.

#### Conclusion

The holdout evaluation demonstrates modest but variable enrichment of positive disclosures among higher-ranked filings.

No model consistently dominated across validation and test periods.

The findings support treating the system as an experimental investigative prioritization tool rather than a validated fraud detection system.

Further work would require additional filing years, more positive examples, careful point-in-time feature auditing, and independent evaluation.


## Step 11: Generate filing-level scores and ranks

In [ ]:
def create_ranked_filings(df, scores_by_model, split):
    result = df[[EIN, YEAR, LABEL]].copy()

    result["split"] = split

    for model_name, scores in scores_by_model.items():

        scores = np.asarray(scores, dtype=float)

        assert len(scores) == len(result), (
            f"Score length mismatch for {model_name}"
        )
        assert np.isfinite(scores).all(), (
            f"Nonfinite scores for {model_name}"
        )

        score_col = f"{model_name}_score"
        rank_col = f"{model_name}_rank"

        result[score_col] = scores

        # Rank 1 = highest priority
        # Stable sorting ensures deterministic tie handling
        order = np.argsort(-scores, kind="stable")

        ranks = np.empty(len(scores), dtype=int)
        ranks[order] = np.arange(1, len(scores) + 1)

        result[rank_col] = ranks

    return result


validation_model_scores = {
    "isolation_forest": iso_val,
    "supervised_weight_50": experiment_scores[("Original", 50)],
    "supervised_weight_500": experiment_scores[("Original", 500)],
    "supervised_oversampled": oversampled_val_scores
}

val_ranked = create_ranked_filings(
    val,
    validation_model_scores,
    "validation_2024_unseen"
)

test_ranked = create_ranked_filings(
    test,
    test_model_scores,
    "test_2025_unseen"
)

print("Validation ranked filings:", val_ranked.shape)
print("Test ranked filings:", test_ranked.shape)

In [ ]:
# Combine validation and test rankings

ranked_filings = pd.concat(
    [val_ranked, test_ranked],
    ignore_index=True
)

# Arrange columns for readability
ranked_columns = [
    EIN,
    YEAR,
    LABEL,
    "split",
    "isolation_forest_score",
    "isolation_forest_rank",
    "supervised_weight_50_score",
    "supervised_weight_50_rank",
    "supervised_weight_500_score",
    "supervised_weight_500_rank",
    "supervised_oversampled_score",
    "supervised_oversampled_rank"
]

ranked_filings = ranked_filings[ranked_columns]

print("Consolidated ranked filing table:")
print("Rows:", len(ranked_filings))
print("Columns:", len(ranked_filings.columns))

display(ranked_filings.head(10))


In [ ]:
# Validate ranked filing table

assert len(ranked_filings) == len(val) + len(test)

assert ranked_filings["split"].value_counts().to_dict() == {
    "validation_2024_unseen": len(val),
    "test_2025_unseen": len(test)
}

model_names = [
    "isolation_forest",
    "supervised_weight_50",
    "supervised_weight_500",
    "supervised_oversampled"
]

for split, group in ranked_filings.groupby("split"):

    n = len(group)

    for model_name in model_names:

        rank_col = f"{model_name}_rank"
        score_col = f"{model_name}_score"

        # Verify ranks are unique and cover 1 through N
        assert group[rank_col].nunique() == n
        assert group[rank_col].min() == 1
        assert group[rank_col].max() == n

        # Verify higher-ranked filings have higher scores
        ordered = group.sort_values(rank_col)

        assert (
            ordered[score_col].diff().dropna() <= 1e-12
        ).all()

print("All ranking checks passed.")

# Inspect top 20 filings under the weight-500 model
display(
    ranked_filings[
        ranked_filings["split"] == "test_2025_unseen"
    ]
    .sort_values("supervised_weight_500_rank")
    .head(20)
)


In [ ]:
# Validate investigative capacity

capacity_rows = []

for split, group in ranked_filings.groupby("split"):

    for model_name in model_names:

        for k in [100, 500, 1000]:

            selected = group.nsmallest(
                k,
                f"{model_name}_rank"
            )

            capacity_rows.append({
                "Split": split,
                "Model": model_name,
                "Top K": k,
                "Positive disclosures found": int(
                    selected[LABEL].sum()
                )
            })

ranked_capacity_check = pd.DataFrame(capacity_rows)

display(
    ranked_capacity_check.pivot_table(
        index=["Split", "Model"],
        columns="Top K",
        values="Positive disclosures found"
    )
)


In [ ]:
# Preserve EIN formatting for export

ranked_filings[EIN] = (
    ranked_filings[EIN]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(9)
)

assert ranked_filings[EIN].str.fullmatch(r"\d{9}").all()

print("EIN formatting verified.")
print("Example EINs:")
display(ranked_filings[[EIN, YEAR]].head(10))


### Step 11D — Consolidated Filing-Level Model Scores and Rankings

#### Objective

Create a unified, filing-level scoring dataset containing the outputs of all four anomaly detection and supervised learning models.

This dataset supports investigative prioritization, model comparison, performance evaluation, and downstream visualization in Databricks.

#### Dataset Construction

The consolidated dataset contains **36,163 filings and 12 columns**:

- **26,114 filings** from the organization-disjoint 2024 validation dataset.
- **10,049 filings** from the organization-disjoint 2025 holdout dataset.

Each filing includes:

- Employer Identification Number (EIN)
- Tax year
- Significant diversion disclosure label (`fraud_label`)
- Evaluation split
- Isolation Forest anomaly score and rank
- Supervised weight-50 score and rank
- Supervised weight-500 score and rank
- Oversampled supervised score and rank

The consolidated dataset includes only validation and holdout filings. Training records are excluded because they were used to fit the models.

EINs are preserved as nine-character strings, including leading zeros, for consistent identification and downstream processing.

#### Models Included

Four previously trained models were used to generate filing-level scores:

1. **Isolation Forest:** Unsupervised anomaly detection model trained to identify statistically unusual financial and organizational characteristics.
2. **Supervised gradient boosting (weight 50):** HistGradientBoostingClassifier trained with a positive class weight of 50.
3. **Supervised gradient boosting (weight 500):** HistGradientBoostingClassifier trained with a positive class weight of 500.
4. **Oversampled supervised gradient boosting:** HistGradientBoostingClassifier trained using random oversampling of the positive class.

All four models use the original 43-feature dataset, transformed into 67 processed columns through median imputation, missingness indicators, and robust scaling.

The additional engineered features tested in Step 8 were not retained because they did not improve validation performance.

No models were retrained or modified during Step 11.

#### Ranking Methodology

Each model assigns a score to every filing.

Filings are ranked in descending score order, with **rank 1 representing the highest-priority filing** under that model.

Rankings are calculated separately within the 2024 validation and 2025 holdout populations.

Consequently, a rank of 1 in 2024 is not directly comparable to a rank of 1 in 2025.

**Score interpretation:**

- Supervised model scores are uncalibrated model outputs used for relative ranking. They should not be interpreted as verified probabilities of fraud or significant diversion.
- Isolation Forest scores represent relative statistical unusualness. Higher anomaly scores indicate that a filing is more unusual according to the model, not that misconduct has occurred.
- Scores from different models are not necessarily on comparable scales. Model ranks provide a more interpretable basis for comparing prioritization.

Ties are resolved using stable sorting in the original row order. Rankings are deterministic for a fixed input order.

#### Data Quality and Validation

All ranking validation checks passed.

The following were verified:

- The consolidated dataset contains the expected 36,163 filings.
- All four models produced finite scores.
- Each model has complete, unique rankings within each evaluation split.
- Ranks range from 1 to the total number of filings in the corresponding split.
- Scores are correctly ordered from highest to lowest priority.
- Top-100, top-500, and top-1,000 positive disclosure counts reproduce the previously calculated evaluation results.

**Investigative-capacity verification:**

| Model | 2024 Found@100 | 2024 Found@500 | 2024 Found@1000 | 2025 Found@100 | 2025 Found@500 | 2025 Found@1000 |
|---|---:|---:|---:|---:|---:|---:|
| Isolation Forest | 0 | 2 | 3 | 0 | 4 | 5 |
| Supervised, weight 50 | 1 | 2 | 2 | 0 | 1 | 3 |
| Supervised, weight 500 | 1 | 3 | 3 | 0 | 1 | 2 |
| Supervised, oversampled | 0 | 3 | 5 | 0 | 2 | 3 |

These results match the investigative-capacity metrics calculated in Steps 9 and 10.

#### Observations

**1. Models produce different investigative rankings.**

The supervised and unsupervised approaches do not necessarily prioritize the same filings.

For example, in the 2025 holdout dataset, EIN `394689461` ranked:

- 1st under Isolation Forest.
- 15th under supervised weight 500.
- 35th under supervised weight 50.
- 47th under the oversampled supervised model.

This demonstrates that different modeling approaches can identify different patterns of statistical unusualness or characteristics associated with significant diversion disclosures.

**2. Highly ranked filings do not necessarily have positive disclosure labels.**

All 20 filings displayed at the top of the 2025 weight-500 supervised ranking had a disclosure label of 0.

This is consistent with the holdout evaluation, where the weight-500 model identified no positive disclosures within its top 100 filings.

However, a label of 0 indicates that no positive significant diversion disclosure was observed in the dataset. It does not establish that the organization is free of fraud, misconduct, or undisclosed diversion.

**3. Investigative yield differs across models and years.**

In 2024 validation, the weight-500 supervised model identified three positive disclosures within the top 500 filings, compared with two for Isolation Forest.

In 2025 holdout, Isolation Forest identified four positive disclosures within the top 500 filings, compared with one for the weight-500 supervised model.

These differences reinforce the importance of evaluating multiple models and reporting results separately by evaluation period.

The 2025 holdout results are reported for generalization assessment, not used to retrospectively change model selection.

#### Intended Use

The consolidated dataset can support:

1. **Investigative prioritization:** Rank nonprofit filings for additional financial or governance review.
2. **Model comparison:** Examine which filings receive high priority under different modeling approaches.
3. **Investigative capacity analysis:** Determine how many positive disclosures appear within the top 100, 500, or 1,000 filings.
4. **Databricks dashboards:** Visualize filing scores, ranks, model comparisons, and disclosure outcomes.
5. **Model agreement analysis:** Identify filings prioritized by multiple models versus filings flagged by only one approach.
6. **Reproducibility:** Preserve the filing-level predictions underlying the evaluation metrics.

For an operational investigative dashboard, historical disclosure labels should be treated as evaluation information rather than predictors or information assumed to be available at scoring time.

#### Important Limitations

**1. The target is not confirmed fraud.**

The disclosure label represents self-reported significant diversion of assets, not independently verified fraud.

A negative disclosure label does not establish that an organization has no diversion or fraud risk.

**2. Rankings are relative to each evaluation population.**

Ranks are calculated separately within the 2024 and 2025 datasets. They should not be interpreted as directly comparable measures of risk across years.

**3. The 2025 filing year is incomplete.**

The holdout dataset may not represent the full population of 2025 filings.

**4. Positive disclosures are extremely rare.**

The validation dataset contains 35 positive disclosures, and the holdout dataset contains only 14.

As a result, model performance estimates are highly sensitive to individual positive cases.

**5. Model scores are not evidence of misconduct.**

A high score or rank indicates that a filing warrants closer examination according to the model's learned patterns.

It does not establish wrongdoing and should not be used as the sole basis for allegations, enforcement decisions, or adverse actions.

**6. Disclosure and reporting bias may affect results.**

Models may learn characteristics associated with an organization's ability or willingness to identify and report significant diversions rather than the underlying occurrence of diversion.

#### Conclusion

Step 11 successfully produced and validated a consolidated filing-level scoring dataset containing **36,163 filings, four model scores, and four corresponding rankings**.

All ranking and investigative-capacity checks passed, confirming consistency with the model evaluation results from Steps 9 and 10.

The dataset provides a reproducible foundation for comparing model recommendations and building investigative-prioritization dashboards in Databricks.

The results support an experimental approach to prioritizing filings for additional review, rather than a validated system for detecting confirmed fraud.

#### Next Step

**Step 12 — Save Models, Scores, and Evaluation Results**

Export the following artifacts for reproducibility and downstream use:

- Consolidated filing-level scores and rankings in Parquet format.
- Validation and holdout evaluation metrics.
- Four fitted model artifacts.
- Fitted preprocessing pipeline.
- Original feature list and model configuration metadata.

These artifacts will support loading the results into Databricks and building the final IRS Form 990 investigative-prioritization demonstration.

## Step 12 — Create Model Output Directories

Create a structured output directory for the final model artifacts, evaluation results, and filing-level rankings.

All saved artifacts correspond to the frozen models evaluated in Steps 9–11. No additional training or tuning is performed.


In [ ]:
OUTPUT_DIR = Path("data/analytics/model_outputs")
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"

for directory in [OUTPUT_DIR, MODELS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directories created:")
print(OUTPUT_DIR.resolve())

In [ ]:
# Save consolidated filing-level scores

RANKED_PATH = OUTPUT_DIR / "form990_ranked_filings_v1.parquet"

ranked_export = ranked_filings.copy()

# Preserve EINs as nine-character strings
ranked_export[EIN] = (
    ranked_export[EIN]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(9)
)

assert ranked_export[EIN].str.fullmatch(r"\d{9}").all()
assert len(ranked_export) == 36163
assert ranked_export.shape[1] == 12

ranked_export.to_parquet(
    RANKED_PATH,
    index=False
)

print("Saved:", RANKED_PATH)
print("Rows:", len(ranked_export))
print("Columns:", len(ranked_export.columns))


In [ ]:
# Save evaluation metrics

VALIDATION_METRICS_PATH = (
    METRICS_DIR / "validation_2024_metrics.parquet"
)

TEST_METRICS_PATH = (
    METRICS_DIR / "holdout_2025_metrics.parquet"
)

ALL_METRICS_PATH = (
    METRICS_DIR / "all_evaluation_metrics.parquet"
)

metrics.to_parquet(
    VALIDATION_METRICS_PATH,
    index=False
)

test_metrics.to_parquet(
    TEST_METRICS_PATH,
    index=False
)

all_evaluation_metrics = pd.concat(
    [metrics, test_metrics],
    ignore_index=True
)

all_evaluation_metrics.to_parquet(
    ALL_METRICS_PATH,
    index=False
)

print("Validation metric rows:", len(metrics))
print("Holdout metric rows:", len(test_metrics))
print("Combined metric rows:", len(all_evaluation_metrics))

display(all_evaluation_metrics.head())


In [ ]:
# Save frozen trained models

model_artifacts = {
    "isolation_forest": iso,
    "supervised_weight_50": weight_models[50],
    "supervised_weight_500": weight_models[500],
    "supervised_oversampled": oversampled_model
}

for model_name, model in model_artifacts.items():

    model_path = MODELS_DIR / f"{model_name}.joblib"

    joblib.dump(model, model_path)

    print("Saved:", model_path)

# Save the fitted original 43-feature preprocessor
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor_original.joblib"

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

print("Saved:", PREPROCESSOR_PATH)


In [ ]:
# Save feature and model metadata

# Use the original 43 features used by the fitted preprocessor
feature_list = list(preprocessor.feature_names_in_)

assert len(feature_list) == 43

metadata = {
    "project": "IRS Form 990 Financial Anomaly and Disclosure Prediction",
    "version": "1.0",
    "notebook": "06_leakage_safe_anomaly_detection_and_disclosure_prediction.ipynb",

    "target_column": LABEL,
    "ein_column": EIN,
    "year_column": YEAR,

    "target_definition": (
        "Self-reported significant diversion of assets; "
        "not independently verified fraud."
    ),

    "training_year": 2023,
    "validation_year": 2024,
    "holdout_year": 2025,

    "training_rows": int(len(train)),
    "validation_rows": int(len(val)),
    "holdout_rows": int(len(test)),

    "training_positives": int(np.asarray(y_train).sum()),
    "validation_positives": int(np.asarray(y_val).sum()),
    "holdout_positives": int(np.asarray(y_test).sum()),

    "original_feature_count": len(feature_list),
    "original_features": feature_list,

    "preprocessing": {
        "imputation": "median",
        "missing_indicators": True,
        "keep_empty_features": True,
        "scaling": "RobustScaler",
        "fit_on": "2023 training data only"
    },

    "models": {
        "isolation_forest": {
            "type": "IsolationForest",
            "score_direction": "higher = more anomalous"
        },
        "supervised_weight_50": {
            "type": "HistGradientBoostingClassifier",
            "positive_class_weight": 50
        },
        "supervised_weight_500": {
            "type": "HistGradientBoostingClassifier",
            "positive_class_weight": 500
        },
        "supervised_oversampled": {
            "type": "HistGradientBoostingClassifier",
            "training_method": "random positive-class oversampling"
        }
    },

    "model_selection": {
        "selection_split": "2024 validation",
        "weight_500": "Selected for approximately 250–500 reviews",
        "oversampled": "Selected for approximately 1,000 reviews",
        "holdout_used_for_selection": False
    },

    "limitations": [
        "Target is self-reported significant diversion, not confirmed fraud.",
        "Negative labels are not verified absence of misconduct.",
        "2025 is an incomplete filing year.",
        "Positive disclosures are extremely rare.",
        "Scores are not calibrated probabilities of fraud.",
        "Rankings are calculated separately within each evaluation split."
    ]
}

METADATA_PATH = OUTPUT_DIR / "model_metadata_v1.json"

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", METADATA_PATH)
print("Original features:", len(feature_list))


In [ ]:
# Final artifact verification

expected_files = [
    RANKED_PATH,
    VALIDATION_METRICS_PATH,
    TEST_METRICS_PATH,
    ALL_METRICS_PATH,
    PREPROCESSOR_PATH,
    METADATA_PATH,
    *[
        MODELS_DIR / f"{name}.joblib"
        for name in model_artifacts
    ]
]

for path in expected_files:
    assert path.exists(), f"Missing artifact: {path}"
    assert path.stat().st_size > 0, f"Empty artifact: {path}"

# Verify metadata
with open(METADATA_PATH) as f:
    loaded_metadata = json.load(f)

assert loaded_metadata["original_feature_count"] == 43

# Verify saved models can be loaded
loaded_models = {
    name: joblib.load(MODELS_DIR / f"{name}.joblib")
    for name in model_artifacts
}

loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

# Confirm loaded models reproduce a small sample of saved scores
sample_n = min(100, len(Xh))
sample_X = Xh[:sample_n]

for name, model in loaded_models.items():
    if name == "isolation_forest":
        reproduced = -model.decision_function(sample_X)
    else:
        reproduced = model.predict_proba(sample_X)[:, 1]

    expected = np.asarray(test_model_scores[name])[:sample_n]

    assert np.allclose(reproduced, expected), (
        f"Prediction mismatch: {name}"
    )

assert (
    list(loaded_preprocessor.feature_names_in_)
    == feature_list
)

print("All artifacts saved and verified successfully.")
print("Artifact count:", len(expected_files))

display(
    pd.DataFrame([
        {
            "file": str(path),
            "size_mb": round(path.stat().st_size / 1024**2, 3)
        }
        for path in expected_files
    ])
)


### Model Artifact Export and Reproducibility

#### Objective

Save the final filing-level scoring dataset, evaluation results, fitted models, preprocessing pipeline, and model metadata for reproducibility and downstream use in Databricks.

No additional model training, tuning, or selection was performed during this step.

#### Exported Artifacts

The following artifacts were saved under `data/analytics/model_outputs/`:

**Filing-Level Scores**
- `form990_ranked_filings_v1.parquet`

Contains 36,163 filings and 12 columns, including identifiers, disclosure labels, evaluation splits, four model scores, and four corresponding ranks.

**Evaluation Metrics**
- `metrics/validation_2024_metrics.parquet`
- `metrics/holdout_2025_metrics.parquet`
- `metrics/all_evaluation_metrics.parquet`

Contains PR-AUC, ROC-AUC, precision, recall, and positive disclosure counts at investigative capacities of 25, 50, 100, 250, 500, and 1,000 filings.

**Trained Models**
- `models/isolation_forest.joblib`
- `models/supervised_weight_50.joblib`
- `models/supervised_weight_500.joblib`
- `models/supervised_oversampled.joblib`

These files preserve the fitted models used in the final evaluation.

**Preprocessing**
- `models/preprocessor_original.joblib`

Preserves the training-fitted median imputation, missingness indicators, and robust scaling transformations applied to the original 43 features.

**Metadata**
- `model_metadata_v1.json`

Documents feature names, data splits, model configurations, target definition, preprocessing, model-selection decisions, and limitations.

#### Reproducibility Verification

The saved artifacts were checked for existence and nonzero file size.

The saved metadata and preprocessing pipeline were successfully reloaded.

All four saved models were reloaded and verified to reproduce their original predictions on a sample of holdout filings.

The consolidated scoring dataset was successfully exported and reloaded from Parquet.

#### Model Selection Integrity

Model selection was performed using the 2024 validation dataset.

The 2025 holdout dataset was used only for final evaluation.

The original 43-feature models were retained following the feature-engineering experiments.

No model selection decisions were changed based on holdout performance.

#### Databricks Integration

The exported Parquet datasets can be loaded into Databricks as Delta tables for downstream analytics.

Potential applications include:

- Investigative-prioritization dashboards.
- Model ranking comparisons.
- Investigative capacity analysis.
- Model agreement and disagreement analysis.
- Historical disclosure evaluation.
- Reproducible model inference.

Historical disclosure labels should be separated from operational scoring inputs and should not be treated as information available when making prospective predictions.

#### Conclusion

The IRS Form 990 anomaly detection and significant diversion disclosure prediction workflow has been completed through model evaluation, filing-level scoring, and artifact export.

The final outputs provide a reproducible foundation for an experimental investigative-prioritization demonstration in Databricks.

Model outputs represent statistical prioritization signals, not verified evidence of fraud or misconduct.



## Final Interpretation, Limitations, and Future Research

### Project Summary

This notebook developed and evaluated four models for identifying statistically unusual IRS Form 990 filings and prioritizing filings associated with self-reported significant diversions of assets.

The workflow included:

1. Leakage-aware feature selection and preprocessing.
2. Organization-disjoint training, validation, and holdout datasets.
3. Unsupervised anomaly detection using Isolation Forest.
4. Supervised gradient boosting with alternative class weights.
5. Random oversampling to address extreme class imbalance.
6. Feature-engineering experiments and model comparisons.
7. Evaluation using PR-AUC, ROC-AUC, precision, recall, and investigative-capacity metrics.
8. Final holdout evaluation without additional model tuning.
9. Consolidated filing-level scoring and ranking.
10. Export and verification of reproducible model artifacts.

The final scoring dataset contains 36,163 validation and holdout filings, with scores and ranks from all four models.

### Key Findings

**1. Significant diversion disclosures are extremely rare.**

The 2024 validation dataset contains 35 positive disclosures among 26,114 filings.

The 2025 holdout dataset contains 14 positive disclosures among 10,049 filings.

This makes conventional classification accuracy a poor measure of model usefulness. Ranking metrics and investigative-capacity analysis are more informative.

**2. No model consistently dominated across evaluation periods.**

The weight-500 supervised model identified three positive disclosures within the top 500 validation filings.

Isolation Forest identified four positive disclosures within the top 500 holdout filings.

The oversampled supervised model achieved the highest holdout ROC-AUC (0.74586).

These results demonstrate variability in model performance and the importance of evaluating multiple approaches.

**3. Feature engineering did not improve validation performance.**

The additional engineered features tested in Step 8 did not improve the selected validation metrics.

The original 43-feature models were therefore retained.

**4. Supervised and unsupervised models identify different patterns.**

The models frequently assign different priorities to the same filings.

This creates opportunities to investigate model agreement, disagreement, and the financial characteristics driving unusual rankings.

**5. Holdout results must not be used for retrospective model selection.**

The 2024 validation dataset was used to select model configurations.

The 2025 holdout dataset was reserved for final evaluation.

Although Isolation Forest performed best at several holdout investigative capacities, this result was not used to change the previously selected models.

### Important Limitations

**Target definition:** The positive label represents a self-reported significant diversion of assets, not independently confirmed fraud.

**Incomplete labels:** A negative disclosure label does not establish that an organization has no fraud or diversion risk.

**Small positive population:** The limited number of positive disclosures makes performance estimates highly uncertain and sensitive to individual filings.

**Incomplete holdout year:** The 2025 dataset may not represent the full population of 2025 Form 990 filings.

**Organization-disjoint evaluation:** The current evaluation focuses on organizations not represented in the training population. Performance for returning organizations remains untested.

**Temporal availability:** Before prospective deployment, all included financial and organizational features must be verified as available at the intended scoring time. Filing-time information must not be treated as available before filing publication.

**Uncalibrated scores:** Supervised model scores are not calibrated probabilities of fraud. Isolation Forest scores represent statistical unusualness rather than evidence of misconduct.

**Disclosure bias:** The models may learn characteristics associated with detecting or reporting significant diversions rather than the occurrence of misconduct itself.

### Recommended Future Research

**1. Investigate highly ranked filings and missed disclosures.**

Review the financial characteristics of top-ranked filings and positive disclosures missed by the models.

Examine whether high scores are associated with unusual financial ratios, governance characteristics, missing data, or reporting patterns.

Avoid interpreting individual scores as allegations of misconduct.

**2. Evaluate returning organizations separately.**

Develop a second evaluation track for organizations with historical filings.

Use strictly backward-looking features and organization-level temporal controls to avoid leakage.

**3. Strengthen point-in-time feature validation.**

Audit when each feature becomes available.

Where reliable publication dates exist, implement as-of-date cutoffs so that prospective predictions use only information available at scoring time.

**4. Explore additional Form 990 series and cross-form relationships.**

After validating temporal availability and data quality, investigate whether supplementary Form 990-EZ, 990-PF, 990-T, or cross-form linkage features improve performance.

**5. Evaluate model stability across additional filing years.**

As more complete data become available, assess whether performance remains consistent across different years, organization types, financial sizes, and disclosure prevalence.

**6. Investigate model agreement and interpretability.**

Compare the top-ranked filings across models and analyze which financial or governance features contribute to their prioritization.

**7. Evaluate probability calibration and alternative modeling approaches.**

If calibrated probabilities of disclosure are needed, evaluate calibration using appropriate training and validation data.

Any new model development should use a newly defined evaluation protocol rather than repeatedly tuning against the existing 2025 holdout results.

### Next Phase — Model Diagnostics and Improvement

Although the modeling workflow has been completed and all artifacts have been saved successfully, predictive performance remains limited and inconsistent across evaluation periods.

The current results do not establish sufficient reliability for operational investigative prioritization.

Before building the final Databricks demonstration, the next phase will focus on understanding the causes of weak model performance and identifying opportunities for improvement.

Planned activities include:

1. Analyze high-ranked filings, missed positive disclosures, and differences between model rankings.
2. Investigate whether model predictions are disproportionately influenced by organization size, missing values, governance characteristics, or financial reporting patterns.
3. Audit the significant diversion disclosure labels for completeness, consistency, and potential reporting bias.
4. Investigate opportunities to expand the training dataset with additional genuine positive examples.
5. Evaluate additional financial, governance, and historical features after validating their availability at the intended scoring time.
6. Compare alternative modeling approaches using an appropriate validation strategy.
7. Establish a fresh evaluation protocol for future model development without repeatedly tuning against the existing 2025 holdout.

The existing models and saved artifacts will be preserved as reproducible baselines.

Once the diagnostic work is complete, the project can proceed to a Databricks demonstration that accurately communicates both the analytical capabilities and limitations of the models.

### Final Conclusion

The project successfully established a reproducible workflow for evaluating financial anomalies and self-reported significant diversion disclosures in IRS Form 990 data.

The results demonstrate some enrichment of positive disclosures among prioritized filings, but performance varies across models and evaluation periods.

The system should be treated as an experimental research and investigative-prioritization capability, not a validated fraud detection system.

The completed modeling outputs provide a foundation for the next phase: an interactive Databricks demonstration.
